# Notebook 03 — Gene × Motif Association Testing (with stability selection)

## Research question
**Does neuronal gene expression predict participation in specific connectome
motifs, beyond what degree alone predicts?**

This is the first analysis in the stack with enough power (N≈170) to produce
a defensible *positive or negative* answer. The old analyses in `archive/` had
critical bugs (dense-edgelist, namespace mismatch, 0 FDR survivors out of 40k
tests); the fixed pipeline is what we test here.

## Hypothesis to pre-register
**H₁**: At least one gene shows Spearman rank-correlation with a degree-residualized
motif participation score (FFL, cycle3, recip, two-step in/out, or clustering) at
Benjamini-Hochberg FDR q < 0.05 across all (gene × motif) tests.

**H₀ (null)**: After FDR correction, zero gene-motif associations survive. The
honest interpretation is: gene expression does not predict motif participation
in C. elegans at this sample size beyond what degree alone explains.

## Preregistered success criteria

1. **Sample size**: N ≥ 150 neurons with both expression (from Notebook 02)
   and motif features (from canonical adjacency in Notebook 01).
2. **Gene space**: variance-filtered to 500–2000 genes (mid-to-high variance
   across expression-mapped neurons). Documented filter, not "tuned to pass".
3. **Primary success**: ≥ 1 gene-motif pair at BH-FDR q_global < 0.05 across
   all tests.
4. **Permutation validation**: on 100 neuron-shuffle permutations, the 95th
   percentile of false-positive hits at FDR<0.05 is strictly less than the
   number of observed hits. If the null produces as many "hits" as the real
   data, there is no real signal.
5. **Stability-selected genes**: for any motif feature with ≥ 3 FDR survivors,
   run 100-bootstrap Lasso stability selection on that feature. Report genes
   with selection frequency ≥ 0.7 (robust predictors).

## Halting rule
If criterion 3 fails: declare null, skip stability selection, document result.
This is scientifically valid and preregistered — it's not a failure of the
notebook, it's a finding.

If criterion 3 passes but 4 fails (permutation null matches the real signal):
the observed hits are dominated by multiple-testing noise. Declare a secondary
null and report.

If both 3 and 4 pass: proceed to stability selection.

## Outputs
- `data_derived/motif_features.csv` — per-neuron motif table
- `data_derived/gene_motif_spearman.csv` — all test results
- `data_derived/gene_motif_fdr_survivors.csv` — hits at q < 0.05 (if any)
- `data_derived/gene_motif_permutation_null.csv` — permutation distribution
- `data_derived/gene_motif_stability.csv` — bootstrap selection frequencies
  (only if 3 and 4 both pass)


In [1]:
import sys, time
from pathlib import Path
_HERE = Path.cwd()
if (_HERE / "lib").is_dir(): sys.path.insert(0, str(_HERE))
elif (_HERE.parent / "lib").is_dir(): sys.path.insert(0, str(_HERE.parent))

from lib.motifs import compute_motifs
from lib.reference import load_nt_reference
from lib.paths import DERIVED

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

RNG = np.random.default_rng(42)

# Connectome
adult = np.load(DERIVED / "connectome_adult.npz", allow_pickle=True)
neurons_c = adult["neurons"]
chem_adj = adult["chem_adj"]
gap_adj  = adult["gap_adj"]
print(f"Witvliet adult: {len(neurons_c)} neurons, {int((chem_adj>0).sum())} chem edges")

# Expression (single-cell aligned)
expr_data = np.load(DERIVED / "expression_tpm.npz", allow_pickle=True)
neurons_e = expr_data["neurons"]
genes_wbg = expr_data["genes_wbg"]
tpm = expr_data["tpm"]
genes_csv = pd.read_csv(DERIVED / "expression_genes.csv")
gene_symbols = genes_csv["symbol"].values
print(f"Expression: {len(neurons_e)} neurons x {len(genes_wbg)} genes")

# NT reference
nt = load_nt_reference()
print(f"NT reference: {len(nt.table)} neurons")


Witvliet adult: 222 neurons, 2191 chem edges
Expression: 222 neurons x 13669 genes
NT reference: 302 neurons


## Step 1 — Restrict to neurons with both motif + expression data

In [2]:
# Neurons with expression are those whose TPM row is not all NaN
has_expr = ~np.all(np.isnan(tpm), axis=1)
expressed_neurons = set(neurons_e[has_expr].tolist())
print(f"Neurons with expression data: {len(expressed_neurons)}")

# Intersect with connectome (they should match after Notebook 02 alignment)
common = [n for n in neurons_c if n in expressed_neurons]
print(f"Neurons in both connectome and expression: {len(common)}")

# Build indices in each space
c_idx = np.array([np.where(neurons_c == n)[0][0] for n in common])
e_idx = np.array([np.where(neurons_e == n)[0][0] for n in common])

chem_sub = chem_adj[np.ix_(c_idx, c_idx)]
gap_sub  = gap_adj[np.ix_(c_idx, c_idx)]
tpm_sub  = tpm[e_idx]
neurons_sub = np.asarray(common)
print(f"Subgraph adjacency: {chem_sub.shape}, tpm_sub: {tpm_sub.shape}")


Neurons with expression data: 179
Neurons in both connectome and expression: 179
Subgraph adjacency: (179, 179), tpm_sub: (179, 13669)


## Step 2 — Compute motif features on the restricted subgraph

In [3]:
mf = compute_motifs(neurons_sub, chem_sub, gap_sub, type_set="both")
feats_resid = mf.degree_residualize()
print(f"Motif features: {feats_resid.shape}")
print(feats_resid[["in_deg","out_deg","ffl","cycle3","recip","clust"]].describe().round(2).to_string())
feats_resid.to_csv(DERIVED / "motif_features.csv")

MOTIF_TARGETS = [
    "ffl_resid", "cycle3_resid", "recip_resid", "two_in_resid", "two_out_resid", "clust_resid",
    "total_deg",  # include raw degree — if a gene predicts "being a hub," that's a finding in itself
]
print(f"\nTargets to test: {MOTIF_TARGETS}")


Motif features: (179, 15)
       in_deg  out_deg     ffl  cycle3   recip   clust
count  179.00   179.00  179.00  179.00  179.00  179.00
mean    12.88    12.88   40.94   22.04    5.54    0.46
std      8.47     7.25   37.83   20.66    3.68    0.43
min      0.00     0.00    0.00    0.00    0.00    0.00
25%      7.00     7.00   12.00    7.50    3.00    0.26
50%     11.00    12.00   30.00   17.00    5.00    0.37
75%     16.50    17.00   58.50   30.50    7.00    0.50
max     43.00    34.00  185.00  125.00   19.00    3.47

Targets to test: ['ffl_resid', 'cycle3_resid', 'recip_resid', 'two_in_resid', 'two_out_resid', 'clust_resid', 'total_deg']


/mnt/ssd4tb/Desktop/C-Elegans/New Notebooks/lib/motifs.py:121: RuntimeWarning: invalid value encountered in divide
  clust = np.where(denom > 0, (ffl_anchor_i + cycle3) / denom, 0.0)


## Step 3 — Filter gene space to variance-threshold survivors

In [4]:
# Criterion: gene must be expressed (>0) in >= 10 of the N mapped neurons,
# and have CV > 1.0 across those neurons (gene varies meaningfully).
# This is a single, documented filter, not tuned iteratively.
MIN_NEURONS_EXPRESSED = 10
MIN_CV = 1.0

n_expressed = (tpm_sub > 0).sum(axis=0)
means = np.nanmean(tpm_sub, axis=0)
stds  = np.nanstd(tpm_sub, axis=0)
cv = np.where(means > 0, stds / (means + 1e-9), 0.0)

keep_mask = (n_expressed >= MIN_NEURONS_EXPRESSED) & (cv >= MIN_CV)
n_kept = int(keep_mask.sum())
print(f"Genes passing filter: {n_kept}")
print(f"  expressed in >= {MIN_NEURONS_EXPRESSED} neurons: {(n_expressed >= MIN_NEURONS_EXPRESSED).sum()}")
print(f"  with CV >= {MIN_CV}: {(cv >= MIN_CV).sum()}")

gene_keep = np.where(keep_mask)[0]
tpm_kept = tpm_sub[:, gene_keep]
wbg_kept = genes_wbg[gene_keep]
sym_kept = gene_symbols[gene_keep]
print(f"Filtered matrix: {tpm_kept.shape}")

# Sanity — are the transmitter-marker genes in the filter?
for sym in ["unc-17","unc-25","eat-4","tph-1","cat-2"]:
    hits = np.where(sym_kept == sym)[0]
    print(f"  {sym}: {'present' if len(hits) else 'FILTERED OUT'}")


Genes passing filter: 8634
  expressed in >= 10 neurons: 10604
  with CV >= 1.0: 11553
Filtered matrix: (179, 8634)
  unc-17: present
  unc-25: FILTERED OUT
  eat-4: present
  tph-1: FILTERED OUT
  cat-2: FILTERED OUT


## Step 4 — All-pairs Spearman, BH-FDR across all tests

In [5]:
from scipy.stats import spearmanr

G = tpm_kept.shape[1]
M = len(MOTIF_TARGETS)
print(f"Running {G} genes × {M} targets = {G*M} Spearman tests...")

rows = []
t0 = time.time()
for gi in range(G):
    x = tpm_kept[:, gi]
    for target in MOTIF_TARGETS:
        y = feats_resid[target].values.astype(float)
        r, p = spearmanr(x, y, nan_policy="omit")
        rows.append((wbg_kept[gi], sym_kept[gi], target, float(r), float(p)))
results = pd.DataFrame(rows, columns=["wbgene","symbol","motif","rho","p"])
print(f"done in {time.time()-t0:.1f}s  ({len(results)} rows)")

# BH-FDR across ALL tests (most conservative)
from statsmodels.stats.multitest import multipletests
results["q_global"] = multipletests(results["p"].fillna(1.0).values, method="fdr_bh")[1]

# Per-motif FDR (less conservative but still principled)
results["q_per_motif"] = np.nan
for motif_name in results["motif"].unique():
    mask = results["motif"] == motif_name
    pvals = results.loc[mask, "p"].fillna(1.0).values
    results.loc[mask, "q_per_motif"] = multipletests(pvals, method="fdr_bh")[1]

results = results.sort_values("p").reset_index(drop=True)

print(f"\nResults preview:")
print(results.head(10).to_string())
results.to_csv(DERIVED / "gene_motif_spearman.csv", index=False)


Running 8634 genes × 7 targets = 60438 Spearman tests...


done in 15.6s  (60438 rows)

Results preview:
           wbgene    symbol         motif       rho             p      q_global   q_per_motif
0  WBGene00022534    syx-16  two_in_resid  0.517873  1.137304e-13  6.873639e-09  9.819485e-10
1  WBGene00009949   F53A2.1     total_deg  0.467280  4.275894e-11  9.232278e-07  3.564510e-07
2  WBGene00008582   F08G5.3  two_in_resid  0.466637  4.582685e-11  9.232278e-07  1.978345e-07
3  WBGene00001993     hpd-1     total_deg  0.461118  8.256914e-11  1.247578e-06  3.564510e-07
4  WBGene00012037   T26E3.4  two_in_resid  0.458438  1.094781e-10  1.323327e-06  3.150779e-07
5  WBGene00019249  H28G03.1     total_deg  0.455714  1.454692e-10  1.356487e-06  4.186603e-07
6  WBGene00009799    lgc-47  two_in_resid  0.454972  1.571099e-10  1.356487e-06  3.391217e-07
7  WBGene00000235     baf-1  two_in_resid  0.450600  2.463779e-10  1.861323e-06  4.254453e-07
8  WBGene00010283     ccz-1  two_in_resid  0.443094  5.256446e-10  3.529879e-06  7.564026e-07
9  WBGene00015

In [6]:
# Survivors
surv_global  = results[results["q_global"] < 0.05]
surv_per_motif = results[results["q_per_motif"] < 0.05]
print(f"SURVIVORS at q_global < 0.05     : {len(surv_global)}")
print(f"SURVIVORS at q_per_motif < 0.05  : {len(surv_per_motif)}")
print(f"\nDistribution by motif (q_global<0.05):")
print(surv_global.groupby("motif").size().to_string())
print(f"\nDistribution by motif (q_per_motif<0.05):")
print(surv_per_motif.groupby("motif").size().to_string())

if len(surv_global) > 0:
    print("\nTop 20 by |rho| (q_global<0.05):")
    print(surv_global.reindex(surv_global["rho"].abs().sort_values(ascending=False).index).head(20).to_string())


SURVIVORS at q_global < 0.05     : 2797
SURVIVORS at q_per_motif < 0.05  : 2920

Distribution by motif (q_global<0.05):
motif
clust_resid      374
cycle3_resid     218
ffl_resid        214
recip_resid      194
total_deg        563
two_in_resid     832
two_out_resid    402

Distribution by motif (q_per_motif<0.05):
motif
clust_resid       334
cycle3_resid      105
ffl_resid          44
recip_resid        74
total_deg         708
two_in_resid     1249
two_out_resid     406

Top 20 by |rho| (q_global<0.05):
            wbgene     symbol         motif       rho             p      q_global   q_per_motif
0   WBGene00022534     syx-16  two_in_resid  0.517873  1.137304e-13  6.873639e-09  9.819485e-10
1   WBGene00009949    F53A2.1     total_deg  0.467280  4.275894e-11  9.232278e-07  3.564510e-07
2   WBGene00008582    F08G5.3  two_in_resid  0.466637  4.582685e-11  9.232278e-07  1.978345e-07
3   WBGene00001993      hpd-1     total_deg  0.461118  8.256914e-11  1.247578e-06  3.564510e-07
4   WBGene

## Step 5 — Permutation null (100 shuffles of neuron→expression mapping)

In [7]:
# For each permutation: shuffle row order of tpm_kept, rerun Spearman on all targets,
# count hits at q_global < 0.05. This tells us the null rate of spurious hits.
N_PERM = 100
N_PERM_QUICK = N_PERM  # full run for a rigorous null

print(f"Running {N_PERM_QUICK} permutations x {G*M} tests each...")
t0 = time.time()
null_hits = []
for perm_i in range(N_PERM_QUICK):
    perm_idx = RNG.permutation(tpm_kept.shape[0])
    tpm_perm = tpm_kept[perm_idx]
    pvals = np.zeros(G * M)
    k = 0
    for target in MOTIF_TARGETS:
        y = feats_resid[target].values.astype(float)
        # vectorized spearman: rank-transform then pearson
        yr = pd.Series(y).rank().values
        for gi in range(G):
            xr = pd.Series(tpm_perm[:, gi]).rank().values
            if np.std(xr) > 0 and np.std(yr) > 0:
                r = np.corrcoef(xr, yr)[0, 1]
                # two-sided p via t approx
                n = len(xr); t = r * np.sqrt(n-2) / np.sqrt(max(1e-12, 1 - r*r))
                from scipy.stats import t as tdist
                p = 2 * tdist.sf(abs(t), df=n-2)
            else:
                p = 1.0
            pvals[k] = p; k += 1
    q_all = multipletests(pvals, method="fdr_bh")[1]
    null_hits.append(int((q_all < 0.05).sum()))
    if (perm_i+1) % 10 == 0:
        print(f"  {perm_i+1}/{N_PERM_QUICK} perms done, current null hits: {null_hits[-5:]} (last 5)")

null_hits = np.array(null_hits)
print(f"\ndone in {time.time()-t0:.1f}s")
print(f"Permutation null FDR-survivor distribution:")
print(f"  mean: {null_hits.mean():.2f}")
print(f"  median: {np.median(null_hits):.1f}")
print(f"  95th percentile: {np.percentile(null_hits, 95):.1f}")
print(f"  max: {null_hits.max()}")
print(f"\nObserved real hits (q_global<0.05): {len(surv_global)}")

pd.Series(null_hits, name="n_fdr_hits").to_csv(DERIVED / "gene_motif_permutation_null.csv", index=False)


Running 100 permutations x 60438 tests each...


  10/100 perms done, current null hits: [0, 0, 0, 0, 0] (last 5)


  20/100 perms done, current null hits: [0, 0, 0, 0, 0] (last 5)


  30/100 perms done, current null hits: [0, 0, 0, 0, 0] (last 5)


  40/100 perms done, current null hits: [0, 0, 0, 2, 0] (last 5)


...

  60/100 perms done, current null hits: [0, 0, 0, 0, 0] (last 5)


  70/100 perms done, current null hits: [0, 0, 0, 0, 0] (last 5)


  80/100 perms done, current null hits: [0, 0, 0, 0, 0] (last 5)


  90/100 perms done, current null hits: [0, 0, 0, 0, 0] (last 5)


  100/100 perms done, current null hits: [0, 0, 0, 0, 0] (last 5)

done in 1167.3s
Permutation null FDR-survivor distribution:
  mean: 0.14
  median: 0.0
  95th percentile: 0.0
  max: 8

Observed real hits (q_global<0.05): 2797


## Step 6 — Preregistered criteria check

In [8]:
N = len(common)
n_genes_tested = G
n_hits = len(surv_global)
null_95 = float(np.percentile(null_hits, 95))

checks = [
    ("1 N >= 150",                               N >= 150,                         f"N={N}"),
    ("2 gene_space in [500, 2000]",              500 <= n_genes_tested <= 2000,    f"{n_genes_tested}"),
    ("3 FDR survivors >= 1",                     n_hits >= 1,                      f"{n_hits} hits at q<0.05"),
    ("4 observed hits > null 95th pctile",       n_hits > null_95,                 f"real={n_hits} vs null_95={null_95:.1f}"),
]

print("PREREGISTERED CRITERIA (primary tests)")
print("=" * 64)
primary_pass = True
for label, passed, detail in checks:
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {label:44s}  {detail}")
    if not passed: primary_pass = False
print("=" * 64)
print(f"PRIMARY CRITERIA PASS (1-4): {primary_pass}")


PREREGISTERED CRITERIA (primary tests)
  [PASS] 1 N >= 150                                    N=179
  [FAIL] 2 gene_space in [500, 2000]                   8634
  [PASS] 3 FDR survivors >= 1                          2797 hits at q<0.05
  [PASS] 4 observed hits > null 95th pctile            real=2797 vs null_95=0.0
PRIMARY CRITERIA PASS (1-4): False


## Step 7 — Stability selection (only if criteria 1–4 pass)

In [9]:
if primary_pass:
    from sklearn.linear_model import Lasso
    from sklearn.utils import resample

    # Pick motifs with >= 3 FDR hits for stability selection
    hit_counts = surv_global.groupby("motif").size()
    focal_motifs = hit_counts[hit_counts >= 3].index.tolist()
    print(f"Running stability selection on motifs with >=3 hits: {focal_motifs}")

    N_BOOT = 100
    ALPHA_LASSO = 0.01

    stability_rows = []
    for target in focal_motifs:
        y = feats_resid[target].values.astype(float)
        y_std = (y - y.mean()) / (y.std() + 1e-9)
        X = tpm_kept.copy().astype(np.float64)
        # Standardize features
        X = (X - X.mean(0, keepdims=True)) / (X.std(0, keepdims=True) + 1e-9)

        freq = np.zeros(G, dtype=float)
        n_sub = int(0.7 * X.shape[0])
        for b in range(N_BOOT):
            idx = RNG.choice(X.shape[0], size=n_sub, replace=False)
            try:
                m = Lasso(alpha=ALPHA_LASSO, max_iter=4000, random_state=1000+b).fit(X[idx], y_std[idx])
                freq += (m.coef_ != 0).astype(float)
            except Exception:
                pass
        freq /= N_BOOT

        top_idx = np.argsort(-freq)[:30]
        for rank, gi in enumerate(top_idx, 1):
            stability_rows.append({
                "motif": target, "rank": rank,
                "wbgene": wbg_kept[gi], "symbol": sym_kept[gi],
                "selection_freq": float(freq[gi]),
            })

    stab_df = pd.DataFrame(stability_rows)
    stab_df.to_csv(DERIVED / "gene_motif_stability.csv", index=False)
    print(f"\nStability selection results (top 10 per motif):")
    for target in focal_motifs:
        s = stab_df[stab_df["motif"]==target].head(10)
        print(f"\n--- {target} ---")
        print(s.to_string(index=False))

    # Criterion 5: any gene with selection_freq >= 0.7 across any focal motif?
    high_stability = stab_df[stab_df["selection_freq"] >= 0.7]
    print(f"\nGenes with stability >= 0.7: {len(high_stability)}")
    if len(high_stability) > 0:
        print(high_stability.to_string(index=False))
else:
    print("Primary criteria failed; skipping stability selection.")
    stab_df = None
    high_stability = pd.DataFrame()


Primary criteria failed; skipping stability selection.


## Step 8 — Final verdict & save survivors

In [10]:
print("=" * 64)
print("NOTEBOOK 03 FINAL VERDICT")
print("=" * 64)
print(f"N neurons tested:                  {N}")
print(f"N genes tested:                    {n_genes_tested}")
print(f"Motif targets:                     {len(MOTIF_TARGETS)}")
print(f"Total tests:                       {G*M}")
print(f"FDR survivors (q_global<0.05):     {n_hits}")
print(f"Permutation null 95th pctile:      {null_95:.1f}")
print(f"Real hits > null?:                 {n_hits > null_95}")
if primary_pass:
    print(f"Stability-selected genes (>=0.7):  {len(high_stability)}")
    verdict = "POSITIVE SIGNAL" if len(high_stability) > 0 else "WEAK POSITIVE (FDR yes, stability no)"
else:
    verdict = "NULL RESULT (preregistered null path)"
print(f"\nVERDICT: {verdict}")

if len(surv_global) > 0:
    surv_global.to_csv(DERIVED / "gene_motif_fdr_survivors.csv", index=False)
    print(f"\nSurvivors saved to {DERIVED / 'gene_motif_fdr_survivors.csv'}")


NOTEBOOK 03 FINAL VERDICT
N neurons tested:                  179
N genes tested:                    8634
Motif targets:                     7
Total tests:                       60438
FDR survivors (q_global<0.05):     2797
Permutation null 95th pctile:      0.0
Real hits > null?:                 True

VERDICT: NULL RESULT (preregistered null path)

Survivors saved to /mnt/ssd4tb/Desktop/C-Elegans/New Notebooks/data_derived/gene_motif_fdr_survivors.csv


## Step 9 — Honest-null interpretation notes

If the result was null:

- The negative result is *scientifically valid*, not a bug: at N≈170, with
  CeNGEN-aligned L4-larval expression and Witvliet adult connectome, degree-
  residualized motif participation does not show gene-expression signal
  surviving FDR correction.
- This could reflect: (a) real biology — connectome motifs are determined by
  developmental wiring rather than adult transcription, (b) a modality mismatch
  — L4 expression predicts L4 connectome, not adult, (c) degree residualization
  absorbed most of the signal, (d) motif definitions are not the right targets.
- Downstream Notebook 04 (PhaseD replication) and Notebook 05 (developmental
  rewiring prediction) remain independent of this result; null here does not
  invalidate them.

If the result was positive:

- Genes surviving BOTH FDR AND stability selection AND the permutation null
  are strong candidates for real biological associations with connectome
  topology.
- Cross-check survivors against Loer & Rand NT classifications and known
  neuropeptide/ion-channel catalogs to see if they cluster into functional
  categories.